# 02 Chunking, embeddings, and index releases

## Learning objectives

- preserve headings, code, tables, and provenance while chunking;
- emit MLflow-compatible `page_content`, `doc_uri`, and `chunk_id` fields;
- fail early when embedding profiles are incompatible;
- treat chunking, embedding, index, prompt, and code changes as one release.


In [ ]:
# Notebook preflight — configuration only; this cell makes no cloud request.
import importlib.util
import sys
from pathlib import Path

setup_path = next(
    path
    for parent in (Path.cwd(), *Path.cwd().parents)
    for path in (
        parent / "notebook_setup.py",
        parent / "examples" / "agentic-ops-rag" / "notebook_setup.py",
    )
    if path.is_file()
)
spec = importlib.util.spec_from_file_location("agentic_ops_rag_setup", setup_path)
setup = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = setup
spec.loader.exec_module(setup)
course_root = setup.find_course_root(setup_path.parent)
session = setup.prepare_notebook_environment(course_root)
session.safe_summary()

## Structure is retrieval evidence

Fixed character windows can split a command from its warning or a table header
from its rows. This small structural chunker keeps heading paths and stable
content-derived identifiers. Real indexing logic belongs in packaged code and a
bundle job, not in a production notebook.


In [ ]:
from agentic_ops_rag import structural_chunks

sample_runbook = """
# Checkout recovery

## Evidence

Capture the deployment ID and trace ID before changing production.

## Command example

```text
propose rollback --release RELEASE_ID
```

The command is a proposal and still needs approval.
"""
chunks = structural_chunks(
    sample_runbook,
    document_id="synthetic-checkout-recovery",
    doc_uri="synthetic://runbooks/training/checkout",
    max_characters=300,
)
[chunk.as_mlflow_document() for chunk in chunks]

Each output is directly usable as retriever-span evidence. `doc_uri` and
`chunk_id` live in metadata exactly where MLflow's RAG judges expect them. The
same content and profile produce the same IDs, which makes re-indexing and
evaluation reproducible.


In [ ]:
from aai_core.rag import ChunkingProfile, EmbeddingProfile

chunking = ChunkingProfile(
    name="markdown-structural",
    version="1",
    chunk_size=900,
    chunk_overlap=120,
    parser="agentic_ops_rag.structural_chunks",
)
indexed_embedding = EmbeddingProfile(
    logical_name="operations-embedding",
    provider="foundry",
    model="embedding-deployment-v1",
    dimensions=1536,
    normalized=True,
    version="1",
)
query_embedding = EmbeddingProfile(
    logical_name="operations-embedding",
    provider="foundry",
    model="embedding-deployment-v1",
    dimensions=1536,
    normalized=True,
    version="1",
)
indexed_embedding.assert_compatible(query_embedding)

In [ ]:
# YOUR TURN — TODO: define the proposed query profile for a controlled change.
proposed_query_embedding = EmbeddingProfile(
    logical_name="operations-embedding",
    provider="foundry",
    model="embedding-deployment-v1",
    dimensions=1536,
    normalized=True,
    version="2",
)

In [ ]:
# CHECK YOUR WORK
indexed_embedding.assert_compatible(proposed_query_embedding)
assert proposed_query_embedding.version != indexed_embedding.version
"The changed profile remains index-compatible and has explicit lineage."

In [ ]:
# Reference solution
incompatible = proposed_query_embedding.__class__(
    logical_name="operations-embedding",
    provider="foundry",
    model="different-embedding-space",
    dimensions=3072,
    normalized=True,
    version="3",
)
try:
    indexed_embedding.assert_compatible(incompatible)
except ValueError as error:
    incompatibility_evidence = str(error)
else:
    raise AssertionError("An incompatible embedding profile must fail early")
incompatibility_evidence

## What changes together

An index is not just a hostname. The release evidence includes source version,
parser and chunking profile, embedding space, index schema, access fields, and
evaluation dataset digest. Azure AI Search index creation and Databricks AI
Search index creation remain external platform actions; application releases
reference their configured logical resource.


In [ ]:
retrieval_release = {
    "logical_retriever": "operations-knowledge",
    "chunking": {
        "name": chunking.name,
        "version": chunking.version,
        "size": chunking.chunk_size,
        "overlap": chunking.chunk_overlap,
    },
    "embedding": {
        "logical_name": proposed_query_embedding.logical_name,
        "model": proposed_query_embedding.model,
        "dimensions": proposed_query_embedding.dimensions,
        "version": proposed_query_embedding.version,
    },
    "required_document_fields": [
        "id",
        "content",
        "source_uri",
        "chunk_id",
        "tenant_id",
        "region",
    ],
}
retrieval_release

In [ ]:
RUN_CONNECTED = False
index_readiness = None
if RUN_CONNECTED:
    resources = session.connected_components(allow_network=True)
    retriever = resources["retriever"]
    index_readiness = {
        "provider": retriever.provider,
        "logical_name": retriever.logical_name,
        "native_client_available": retriever.native_client is not None,
    }
index_readiness

## Knowledge check

Answer from the evidence you produced, not from memory:

1. Why is an embedding dimension change an index compatibility event?
2. Which document fields make a retriever span useful to MLflow judges?
3. Why does production chunking live in a job under src rather than a notebook?

<details>
<summary>How to use this check</summary>

If an answer cannot point to a row, trace, contract, or failed check from this
lesson, revisit the exercise before moving on.

</details>


## Recap

You produced stable chunks, inspected their MLflow document shape, and proved an
embedding mismatch fails before an expensive query. Lesson 03 compares managed
retrieval modes without treating their raw score ranges as interchangeable.
